## 0. LangSmith

In [ ]:
# ---- LangSmith observability setup (EU region) ----
import os
from dotenv import load_dotenv

# ---- Load environment variables from .env file ---- 
load_dotenv(override=False)

# Enable LangSmith tracing
os.environ.setdefault("LANGSMITH_TRACING", "true")
langsmith_proj_name_baseline = "field-guide-rag-baseline"
langsmith_proj_name_improved = "field-guide-rag-improved"
os.environ.setdefault("LANGSMITH_PROJECT", langsmith_proj_name_improved) #XXXXX

# ---- Set the European endpoint ----
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

# ---- Ensure the API key is present ---- 
if not os.environ.get("LANGSMITH_API_KEY"):
    from getpass import getpass
    os.environ["LANGSMITH_API_KEY"] = getpass("Enter your LangSmith API key: ")

print(f"LangSmith tracing enabled. Project: {os.environ['LANGSMITH_PROJECT']}")
print(f"Endpoint: {os.environ['LANGSMITH_ENDPOINT']}")

LangSmith tracing enabled. Project: field-guide-rag-improved
Endpoint: https://eu.api.smith.langchain.com


## 1. ChromaDB connection — "Species Accounts" collection


In [ ]:
from pathlib import Path
import shutil
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

PROJECT_DIR = Path.home() / 'Downloads' / 'AgenticFolder' / 'project0001'
PERSIST_DIR = PROJECT_DIR / 'DB Species Accounts'
COLLECTION_NAME = 'db_species_accounts'

# --- Embedding settings ---- 
EMBEDDING_MODEL = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
print(f"Embedding model set: {EMBEDDING_MODEL}")

# ---- Database ---- 
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIR),
)


Embedding model set: text-embedding-3-small


## 2. Image classifier model


In [ ]:
import getpass
import json
import mimetypes
import os
import requests

API_URL = 'https://api.inaturalist.org/v2/computervision/score_image'
USER_AGENT = 'iNaturalist-CV-notebook/1.0 (personal learning project)'

jwt_token = os.environ.get('INAT_JWT')
if not jwt_token:
    jwt_token = getpass.getpass('Paste your iNaturalist API JWT: ').strip()
if not jwt_token:
    raise ValueError('An iNaturalist API JWT is required.')
print('JWT loaded (not displayed).')

def identify_image(image_path, jwt, *, lat=None, lng=None, observed_on=None, top_n=3):
    """Return normalized iNaturalist CV suggestions for one local image."""
    image_path = Path(image_path)
    if not image_path.is_file():
        raise FileNotFoundError(f'Image not found: {image_path}')
    fields = {'combined_score': True, 'vision_score': True,
              'taxon': {'id': True, 'name': True, 'preferred_common_name': True}}
    data = {key: value for key, value in {'lat': lat, 'lng': lng, 'observed_on': observed_on}.items() if value is not None}
    data['fields'] = json.dumps(fields)
    content_type = mimetypes.guess_type(image_path.name)[0] or 'application/octet-stream'
    headers = {'Authorization': f'Bearer {jwt}', 'User-Agent': USER_AGENT}
    with image_path.open('rb') as image_file:
        response = requests.post(API_URL, headers=headers, data=data,
                                 files={'image': (image_path.name, image_file, content_type)}, timeout=60)
    if response.status_code == 429:
        raise RuntimeError('iNaturalist rate-limited this request. Wait before trying again.')
    try:
        response.raise_for_status()
    except requests.HTTPError as exc:
        raise RuntimeError(f'iNaturalist returned {response.status_code}: {response.text[:500]}') from exc
    payload = response.json()
    suggestions = []
    for item in payload.get('results', [])[:top_n]:
        taxon = item.get('taxon') or {}
        suggestions.append({'scientific_name': taxon.get('name', 'Unknown taxon'),
                            'common_name': taxon.get('preferred_common_name'),
                            'taxon_id': taxon.get('id'),
                            'combined_score': item.get('combined_score'),
                            'vision_score': item.get('vision_score')})
    return suggestions, payload

name_of_specie = None


JWT loaded (not displayed).


## 3. RAG pipeline


In [ ]:
# ---- RETRIEVER + DOCUMENT HELPERS ---- 
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.tools import tool
from langchain_community.document_compressors import FlashrankRerank
from flashrank import Ranker

def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)


# ---- Build a citation string from chunk metadata ---- 
def format_citation(metadata: dict) -> str:
    source_type = metadata.get('cite_source_type')
    title = metadata.get('cite_title')
    url = metadata.get('cite_url')

    if not source_type and not title:
        return metadata.get('source', 'Unknown Source')

    if source_type == 'wikipedia':
        label = f'Wikipedia, "{title}"' if title else 'Wikipedia'
        return f'{label} — {url}' if url else label

    if source_type == 'open_library':
        creator = metadata.get('cite_archive_org_creator')
        date = metadata.get('cite_archive_org_date')
        bits = [title or metadata['source']]
        if creator:
            bits.append(creator)
        if date:
            bits.append(str(date)[:4])
        citation = ', '.join(bits)
        notes = metadata.get('cite_manifest_notes')
        if notes:
            citation += f' ({notes})'
        return f'{citation} — {url}' if url else citation

    if source_type in ('europe_pmc', 'arxiv', 'libgen'):
        author = metadata.get('cite_author')
        year = metadata.get('cite_year')
        journal = metadata.get('cite_journal')
        bits = [b for b in (author, title or metadata['source'], journal, str(year) if year else None) if b]
        citation = '. '.join(b.rstrip('.') for b in bits)
        doi = metadata.get('cite_doi')
        if doi:
            citation += f' — https://doi.org/{doi}'
        elif url:
            citation += f' — {url}'
        return citation

    # ---- Any other/unrecognised source_type: show what we have without guessing shape ---- 
    bits = [b for b in (title, metadata.get('cite_author'), metadata.get('cite_year')) if b]
    citation = ', '.join(bits) if bits else metadata.get('source', 'Unknown Source')
    return f'{citation} — {url}' if url else citation


# ---- Get unique formatted citations from docs ---- 
def extract_sources(docs):
    return sorted({format_citation(doc.metadata) for doc in docs})


# ---- baseline variables ---- 
DEFAULT_K_baseline = 15
FINAL_k_baseline = 7
score_threshold_baseline = 0.7

# ---- improved variables ---- 
DEFAULT_K_improved = 30
FINAL_k_improved = 10
score_threshold_improved = 0.6

reranker = FlashrankRerank(score_threshold = score_threshold_improved, top_n=FINAL_k_improved, model="ms-marco-MultiBERT-L-12") #XXXXX

def search_retrieve_rerank(query, retrieve_k=DEFAULT_K_improved, final_k=FINAL_k_improved):
    docs = vector_store.similarity_search(query, k=retrieve_k)
    reranked_docs = reranker.compress_documents(docs, query)
    return reranked_docs[:final_k]


/var/folders/zh/vbl6k4hs1ggfwjfqn6t_gb100000gn/T/ipykernel_6767/4220965452.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_compressors import FlashrankRerank


In [ ]:
# LLM
import os

MODEL_NAME_baseline = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
REASONING_EFFORT_baseline = os.getenv("OPENAI_REASONING_EFFORT", "medium").lower()

MODEL_NAME_improved = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")
REASONING_EFFORT_improved = os.getenv("OPENAI_REASONING_EFFORT", "max").lower()
VALID_REASONING_EFFORTS = {"none", "low", "medium", "high", "xhigh", "max"}
if REASONING_EFFORT_improved not in VALID_REASONING_EFFORTS:
    raise ValueError(
        f"OPENAI_REASONING_EFFORT must be one of "
        f"{sorted(VALID_REASONING_EFFORTS)}; got {REASONING_EFFORT_improved!r}"
    )

ANTHROPIC_MODEL_NAME = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-5")


def _build_llm():
    try:
        candidate = ChatOpenAI(
            model=MODEL_NAME_improved,
            use_responses_api=True,
            timeout=60,
            max_retries=2,
            max_tokens=None,
            reasoning={"effort": REASONING_EFFORT_improved},
        )
        # Cheap call to confirm the key/endpoint actually works before committing to it --
        # ChatOpenAI() succeeds even with a bad/missing key, it only fails on first use.
        candidate.invoke("ping")
        print(f"Using OpenAI model: {MODEL_NAME_improved}")
        return candidate
    except Exception as exc:
        print(f"OpenAI LLM unavailable ({exc!r}); falling back to Anthropic.")
        from langchain_anthropic import ChatAnthropic

        fallback = ChatAnthropic(
            model=ANTHROPIC_MODEL_NAME,
            timeout=60,
            max_retries=2,
            temperature=0,
            max_tokens=1024,
        )
        print(f"Using Anthropic model: {ANTHROPIC_MODEL_NAME}")
        return fallback


llm = _build_llm()

tool_llm = llm.with_config(tags=["internal"])

INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


Using OpenAI model: gpt-5.6-luna


In [ ]:
# ---- PLAIN HELPERS ---- 
import json
import re


def new_chat_history() -> list[dict]:
    """Fresh history seeded with agent_1's system prompt -- agent_1 drives the user-facing
    conversation, so that's what the visible chat is seeded with. Deterministic -- not a tool."""
    return [{"role": "system", "content": SYSTEM_PROMPT_1}]


MUSHROOM_TERMS = ('mushroom', 'toadstool', 'fungus', 'fungi', 'agaric', 'bolete')
TOXIC_TERMS = ('toxic', 'poison', 'venomous', 'dangerous', 'harmful')
INVASIVE_TERMS = ('invasive',)
PROTECTED_TERMS = ('protected',)


def safety_warnings(species: str, question: str, answer: str) -> list[str]:
    """Mandatory post-check appended after every answer, regardless of what either agent said.
    Deliberately NOT a tool -- it must always run, so it can't be something an agent opts into."""
    haystack = f'{species} {question} {answer}'.lower()
    warnings = []
    if any(term in haystack for term in MUSHROOM_TERMS):
        warnings.append('⚠️ **Never eat a wild mushroom** on the strength of an app identification — '
                        'confirm with a qualified expert before any consumption, every time.')
    if any(term in haystack for term in TOXIC_TERMS):
        warnings.append('⚠️ **This may be toxic or otherwise dangerous** — read the notes below before acting.')
    if any(term in haystack for term in INVASIVE_TERMS):
        warnings.append('🚫 **This may be a regionally invasive species** — check local reporting/disposal rules '
                        'before doing anything with it.')
    if any(term in haystack for term in PROTECTED_TERMS):
        warnings.append('🛡️ **This may be a legally protected species** — check regulations before disturbing '
                        'or removing it.')
    seen, ordered = set(), []
    for warning in warnings:
        if warning not in seen:
            seen.add(warning)
            ordered.append(warning)
    return ordered



# ---- SUBJECT RESOLUTION ---- 
_QUESTION_PREFIX_RE = re.compile(
    r"^\s*(?:please\s+)?(?:can you\s+|could you\s+)?"
    r"(?:tell me (?:more )?about|what about|what is|explain|describe)\s+",
    re.IGNORECASE,
)

# ---- a capitalised genus followed by a lowercase epithet ---- 
_BINOMIAL_RE = re.compile(r"\b([A-Z][a-z]{2,})\s+([a-z]{3,})\b")


def extract_topic(question: str) -> str:
    """Deprecated: kept only so older cells/notebooks still import. Strips common leading
    phrases ('tell me more about dogs' -> 'dogs') with no awareness of the conversation.
    Use `resolve_subject(...)['search_query']` instead."""
    if not question:
        return question
    topic = _QUESTION_PREFIX_RE.sub("", question).strip()
    topic = topic.rstrip("?.! ")
    return topic or question


def _history_digest(history, max_turns: int = 4, clip: int = 300) -> str:
    """A short, plain-text view of the conversation for the resolver: the running summary plus
    the last few verbatim turns, each clipped. Reads a HybridMemory (section 4) defensively via
    getattr so this still works if it's handed a plain list or None."""
    if history is None:
        return "(no conversation yet)"
    parts: list[str] = []
    summary = getattr(history, "summary", "")
    if summary:
        parts.append(f"[summary of earlier turns] {summary}")
    recent = getattr(history, "recent_messages", None)
    if recent is None and isinstance(history, list):
        recent = history
    for message in (recent or [])[-max_turns:]:
        role = message.get("role", "?")
        if role == "system":
            continue
        content = (message.get("content") or "").strip().replace("\n", " ")
        parts.append(f"{role}: {content[:clip]}")
    return "\n".join(parts) or "(no conversation yet)"


SUBJECT_RESOLVER_PROMPT = ChatPromptTemplate.from_template(
    """You prepare retrieval queries for a UK wildlife field-guide assistant.

A species was identified from the user's photo, and the user is now chatting about it in
natural language. They will say "it", "this one", "the mushroom"; they may name a completely
different species; they may ask something general with no species behind it at all. Your job
is to work out what the newest message is about and turn it into one self-contained query.

Species currently under discussion: {active_subject}
Region of interest: {region}

Conversation so far:
{context}

Newest user message:
{question}

Reply with ONLY a JSON object -- no prose, no markdown fences -- with exactly these keys:
  "subject": the species the newest message is about, as a scientific name when you know it,
             otherwise the common name. Use the species currently under discussion whenever
             the message refers to it by pronoun or implication. null if the message isn't
             about a particular species.
  "subject_changed": true only if the user has switched to a species other than the one
             currently under discussion; false otherwise.
  "comparison_with": the second species, if the user is asking to compare two; else null.
  "search_query": one self-contained field-guide search query. Every pronoun resolved, the
             subject's name spelled out, plus the topic words from the message. No question
             mark, no "tell me about".
  "is_general": true if this is small talk, a meta question about the assistant, or otherwise
             not answerable from a species field guide.

Examples (with "Vulpes vulpes" under discussion):
  "is it dangerous to my dog?" ->
    {{"subject": "Vulpes vulpes", "subject_changed": false, "comparison_with": null,
      "search_query": "Vulpes vulpes red fox risk to dogs pets aggression behaviour disease",
      "is_general": false}}
  "how does it compare to a badger?" ->
    {{"subject": "Vulpes vulpes", "subject_changed": false, "comparison_with": "Meles meles",
      "search_query": "Vulpes vulpes red fox compared with Meles meles badger habitat diet behaviour",
      "is_general": false}}
  "actually, what about Japanese knotweed?" ->
    {{"subject": "Reynoutria japonica", "subject_changed": true, "comparison_with": null,
      "search_query": "Reynoutria japonica Japanese knotweed identification spread control",
      "is_general": false}}
  "thanks, that's helpful" ->
    {{"subject": "Vulpes vulpes", "subject_changed": false, "comparison_with": null,
      "search_query": "", "is_general": true}}
"""
)

subject_resolver_chain = SUBJECT_RESOLVER_PROMPT | tool_llm | StrOutputParser()

_JSON_FENCE_RE = re.compile(r"^\s*```(?:json)?\s*|\s*```\s*$", re.IGNORECASE)


def _parse_resolver_json(raw: str) -> dict | None:
    """Parse the resolver's reply, tolerating markdown fences and surrounding prose."""
    if not raw:
        return None
    text = _JSON_FENCE_RE.sub("", raw.strip())
    try:
        parsed = json.loads(text)
    except (TypeError, ValueError):
        start, end = text.find("{"), text.rfind("}")
        if start == -1 or end <= start:
            return None
        try:
            parsed = json.loads(text[start:end + 1])
        except (TypeError, ValueError):
            return None
    return parsed if isinstance(parsed, dict) else None


def _fallback_resolution(question: str, active_subject: str | None) -> dict:
    """Regex-only resolution -- the old `extract_topic` behaviour, plus a binomial sniff and a
    fall back to the active species. Used when the resolver LLM call fails."""
    topic = extract_topic(question or "")
    match = _BINOMIAL_RE.search(question or "")
    subject = match.group(0) if match else active_subject
    # Don't repeat the name if the user already typed it into the question.
    if subject and subject.lower() in topic.lower():
        query = topic
    else:
        query = " ".join(part for part in (subject, topic) if part).strip()
    return {
        "subject": subject,
        "subject_changed": bool(match) and bool(active_subject)
                           and match.group(0).lower() != (active_subject or "").lower(),
        "comparison_with": None,
        "search_query": query or (subject or topic),
        "is_general": False,
        "resolved_by": "fallback",
    }


def resolve_subject(question: str | None,
                    active_subject: str | None = None,
                    history=None,
                    region: str | None = None) -> dict:
    """Work out what a user message is about and build a self-contained retrieval query.

    Plain function, not a tool: it runs before agent_1 is invoked, on every turn, to prepare
    that turn's instruction -- it isn't something agent_1 chooses to call. (It does use an LLM
    internally, tagged 'internal' via tool_llm so its tokens never leak into the streamed
    answer -- see _is_agent_1_reply in section 8.)"""
    if not question or not question.strip():
        return {"subject": active_subject, "subject_changed": False, "comparison_with": None,
                "search_query": active_subject or "", "is_general": False,
                "resolved_by": "no-question"}

    try:
        raw = subject_resolver_chain.invoke({
            "question": question.strip(),
            "active_subject": active_subject or "(none -- no photo identified yet)",
            "region": region or "(not specified)",
            "context": _history_digest(history),
        })
        parsed = _parse_resolver_json(raw)
    except Exception:
        parsed = None

    if not parsed:
        return _fallback_resolution(question, active_subject)

    subject = parsed.get("subject") or active_subject
    resolution = {
        "subject": subject,
        "subject_changed": bool(parsed.get("subject_changed")) and bool(subject),
        "comparison_with": parsed.get("comparison_with") or None,
        "search_query": (parsed.get("search_query") or "").strip(),
        "is_general": bool(parsed.get("is_general")),
        "resolved_by": "llm",
    }
    if not resolution["search_query"] and not resolution["is_general"]:
        resolution["search_query"] = _fallback_resolution(question, subject)["search_query"]
    return resolution


def build_user_content(species: str | None, question: str | None,
                       is_initial: bool, resolution: dict | None = None) -> str:
    """Builds the next user turn as an instruction for agent_1. No pre-fetched context is
    embedded -- agent_1 decides whether/how to call retrieve_documents itself. Plain function,
    not a tool: no side effects, nothing here for the LLM to decide.

    `resolution` is the output of `resolve_subject` for this turn. It's what lets the user
    write "is it dangerous?" instead of "is Amanita muscaria dangerous?": the resolved subject
    and search query are handed to agent_1 explicitly, so the species survives into the
    retrieval call even though the user never typed it."""
    if is_initial:
        if not species:
            raise ValueError("An identified species is required for the initial report.")
        return (
            f"Tell me about this species: {species}. What is its status in the UK (if any), and what "
            f"should I do about it? Call retrieve_documents for {species} first, and "
            f"consult_safety_analyst for its conservation/invasive status."
        )

    if not question:
        raise ValueError("Question is required for follow-up messages.")

    resolution = resolution or resolve_subject(question, active_subject=species)
    subject = resolution.get("subject") or species
    search_query = resolution.get("search_query") or question
    lines: list[str] = []

    if subject:
        if resolution.get("subject_changed") and species and subject != species:
            lines.append(
                f"The user has moved the conversation from {species} to {subject}. Answer "
                f"about {subject} from here on; only mention {species} if they ask you to "
                f"relate the two."
            )
        else:
            lines.append(f"Subject of this message: {subject}.")
    else:
        lines.append(
            "No species has been identified yet (the user hasn't uploaded a photo) and this "
            "message doesn't name one. If the field guide might still cover it, retrieve on "
            "the question itself; otherwise answer as a general naturalist assistant and say "
            "that a photo would let you be specific."
        )


    _status_terms = TOXIC_TERMS + INVASIVE_TERMS + PROTECTED_TERMS + (
        "conservation", "legal", "protected status", "endangered", "status",
    )
    question_asks_about_status = any(term in question.lower() for term in _status_terms)

    if resolution.get("is_general"):
        lines.append(
            "This looks like conversational or meta chat rather than a field-guide question. "
            "Reply naturally and briefly; only retrieve if you actually need grounding."
        )
    elif resolution.get("comparison_with"):
        lines.append(
            f"The user is asking for a comparison. Call compare_species with "
            f"'{subject}' and '{resolution['comparison_with']}'."
        )
        lines.append(f"Suggested retrieval query: {search_query}")
        if question_asks_about_status:
            lines.append(
                f"The user is also asking about status -- call consult_safety_analyst for "
                f"{subject or 'the species involved'} and fold that into the comparison."
            )
    else:
        lines.append(
            f"Call retrieve_documents before answering, using this self-contained query "
            f"(the user's own wording may rely on pronouns): {search_query}"
        )
        if question_asks_about_status:
            lines.append(
                f"This question touches on toxicity, invasive status, or legal protection -- "
                f"also call consult_safety_analyst for {subject or 'the species involved'} "
                "and answer using only what these return; if they don't support an answer, "
                "say so."
            )
        else:
            lines.append(
                "This question is not about conservation/invasive/legal status -- do not call "
                "consult_safety_analyst and do not include a conservation/invasive-status "
                "section unless the user actually asked about it. Answer only what was asked."
            )

    lines.append(f"\nUser question (verbatim): {question}")
    return "\n".join(lines)


In [ ]:
# ---- TOOLS FOR AGENT 2 (safety_analyst) ---- 
@tool
def extract_conservation_status(species: str) -> dict:
    """Extract conservation-status mentions (IUCN Red List category, UK legal protection) for a
    species from the field-guide database."""
    docs = search_retrieve_rerank(species, retrieve_k=DEFAULT_K_improved, final_k=FINAL_k_improved)  # fixed: was retriever.invoke(species, search_kwargs=...)
    if not docs:
        return {"content": "No documents found for this species.", "sources": []}
    patterns = [
        r'(IUCN|Red List|conservation status|protected|Priority species|BAP|Section 41)',
        r'(Vulnerable|Endangered|Critically Endangered|Near Threatened|Least Concern|Data Deficient)',
        r'(protected in the UK|Schedule \d+|Wildlife and Countryside Act)',
    ]
    findings = []
    for doc in docs:
        text = doc.page_content
        for pat in patterns:
            findings.extend(re.findall(pat, text, re.IGNORECASE))
    if findings:
        content = f"Conservation status mentions: {', '.join(sorted(set(findings)))}"
    else:
        content = "No explicit conservation status found in the retrieved documents."
    return {"content": content, "sources": extract_sources(docs)}


# ---- Small reference table for species with well-known invasive status in the UK ---- 
INVASIVE_DB = {
    ("Reynoutria japonica", "UK"): "Highly invasive; spread by rhizomes.",
    ("Fallopia japonica", "UK"): "Highly invasive; spread by rhizomes.",
    ("Impatiens glandulifera", "UK"): "Highly invasive; Himalayan balsam, outcompetes native riverbank flora and spreads explosively via seed dispersal.",
    ("Heracleum mantegazzianum", "UK"): "Highly invasive and hazardous; giant hogweed, sap causes severe skin photosensitivity and burns.",
    ("Crassula helmsii", "UK"): "Highly invasive aquatic plant; New Zealand pygmyweed, forms dense mats that choke ponds and waterways.",
    ("Hydrocotyle ranunculoides", "UK"): "Highly invasive aquatic plant; floating pennywort, rapid growth blocks waterways and depletes oxygen.",
    ("Rhododendron ponticum", "UK"): "Highly invasive shrub; forms dense thickets, shades out native woodland flora, and hosts Phytophthora pathogens.",
    ("Lagarosiphon major", "UK"): "Highly invasive aquatic plant; curly waterweed, outcompetes native submerged vegetation.",
    ("Elodea canadensis", "UK"): "Invasive aquatic plant; Canadian waterweed, can dominate slow-moving freshwater bodies.",
    ("Sciurus carolinensis", "UK"): "Invasive; grey squirrel, outcompetes native red squirrel and carries squirrelpox virus.",
    ("Oxyura jamaicensis", "UK"): "Invasive; ruddy duck, threatens the native white-headed duck through hybridisation (subject to eradication programme).",
    ("Myocastor coypus", "UK"): "Formerly invasive and eradicated in the UK (by 1989); coypu, damaged riverbanks and wetland vegetation.",
    ("Neovison vison", "UK"): "Invasive; American mink, major predator of water voles and ground-nesting birds.",
    ("Procyon lotor", "UK"): "Not established but listed as invasive non-native under monitoring; raccoon, potential threat to native wildlife if it establishes.",
    ("Trachemys scripta", "UK"): "Invasive; red-eared terrapin, competes with native species in ponds and wetlands where released.",
    ("Pacifastacus leniusculus", "UK"): "Highly invasive; signal crayfish, spreads crayfish plague fatal to the native white-clawed crayfish and burrows into riverbanks.",
    ("Dreissena polymorpha", "UK"): "Highly invasive; zebra mussel, fouls infrastructure and outcompetes native mussels in freshwater systems.",
    ("Harmonia axyridis", "UK"): "Invasive; harlequin ladybird, outcompetes and preys on native ladybird species.",
    ("Vespa velutina", "UK"): "Highly invasive; Asian hornet, preys on honeybees and other pollinators; subject to active surveillance and eradication.",
    ("Ondatra zibethicus", "UK"): "Formerly invasive and eradicated in the UK (by 1930s); muskrat, damaged riverbanks through burrowing.",
    # add more entries as your field guide's corpus grows
}


@tool
def check_invasive_status(species: str, region: str = "UK") -> dict:
    """Check if a species is considered invasive in a given region. Checks a small reference
    table first, then falls back to a keyword search of the field guide for that species."""
    key = (species, region)
    if key in INVASIVE_DB:
        return {"content": f"Invasive status in {region}: {INVASIVE_DB[key]}",
                "sources": ["INVASIVE_DB reference table"]}
    docs = search_retrieve_rerank(f"{species} invasive {region}", retrieve_k=DEFAULT_K_improved, final_k=FINAL_k_improved)
    for doc in docs:
        if 'invasive' in doc.page_content.lower():
            return {"content": (f"No entry in the reference table, but the field guide mentions "
                                f"'invasive' in connection with {species}: "
                                f"\"{doc.page_content[:300]}...\""),
                    "sources": extract_sources([doc])}
    return {"content": f"No invasive-status information found for '{species}' in '{region}'.",
            "sources": []}


TOOLS_2 = [
    extract_conservation_status,
    check_invasive_status,
]

print([t.name for t in TOOLS_2])


['extract_conservation_status', 'check_invasive_status']


In [ ]:
# ---- BUILD AGENT 2 - safety_analyst ---- 
from langchain.agents import create_agent

SYSTEM_PROMPT_2 = """You are a safety advisor naturalist assistant, consulted by another agent
(not by the end user directly), only when that agent has decided a status check is actually
needed for this turn. Provide clear, cited, and actionable status checks.
If the retrieved context does not support an answer, clearly state that. Do not claim any
image-based identification is verified.

Your tools:
- extract_conservation_status: finds conservation-status mentions (IUCN, UK legal status) in the
  field-guide database for a species.
- check_invasive_status: checks whether a species is flagged as invasive in a given region,
  falling back to a keyword search of the field guide if it isn't in the reference table.

Keep your answer to a few sentences -- you're producing an input for another agent's answer,
not the final response the user sees. The calling agent will decide whether and how to include
what you return; don't add recommendations, disclaimers, or external links of your own, just
report the status findings plainly.
"""

agent_2 = create_agent(
    model=tool_llm,
    tools=TOOLS_2,
    system_prompt=SYSTEM_PROMPT_2,
    name="safety_analyst",
)

print(f"Agent 2 (safety_analyst) ready with {MODEL_NAME_improved} and {len(TOOLS_2)} tools ✅")


Agent 2 (safety_analyst) ready with gpt-5.6-luna and 2 tools ✅


In [ ]:
# ---- TOOLS FOR AGENT 1 (wildlife_analyst) ---- 
import json


@tool
def retrieve_documents(query: str) -> dict:
    """Retrieve grounding passages from the field-guide vector store for a query. Call this
    before answering any factual question about a species' identification, status, ecology,
    toxicity, or legal protection -- answer only using what this returns."""
    docs = search_retrieve_rerank(query,  retrieve_k=DEFAULT_K_improved, final_k=FINAL_k_improved)
    if not docs:
        return {"content": "No documents found.", "sources": []}
    return {"content": format_docs(docs), "sources": extract_sources(docs)}



@tool
def summarise_topic(query: str) -> dict:
    """Return a insighful text (with maximum of 2000 words) and final summary at the end of what the field guide says about a topic or species."""
    docs = search_retrieve_rerank(query, retrieve_k=DEFAULT_K_improved, final_k=FINAL_k_improved)
    if not docs:
        return {"content": "No relevant documents found.", "sources": []}
    combined = "\n".join(d.page_content for d in docs)
    prompt = ChatPromptTemplate.from_messages([
        ("human", "Based on the following text, produce an insightful account of at most "
                  "2000 words, ending with a concise summary of the most important "
                  "facts:\n\n{combined}")
    ])
    summary = (prompt | tool_llm | StrOutputParser()).invoke({"combined": combined})
    return {"content": summary, "sources": extract_sources(docs)}




@tool
def compare_species(species1: str, species2: str) -> dict:
    """Compare two species (habitat, diet, behaviour, threats, conservation status) using the
    field guide's data on each."""
    docs1 = search_retrieve_rerank(species1, retrieve_k=DEFAULT_K_improved, final_k=FINAL_k_improved)
    docs2 = search_retrieve_rerank(species2, retrieve_k=DEFAULT_K_improved, final_k=FINAL_k_improved)
    context1 = "\n".join(d.page_content for d in docs1)
    context2 = "\n".join(d.page_content for d in docs2)
    prompt = ChatPromptTemplate.from_messages([("human", """Compare the following two species based on the provided context.
Species 1: {species1}
Context: {context1}
Species 2: {species2}
Context: {context2}
Provide a balanced comparison covering habitat, diet, behaviour, threats, and conservation status.""")])
    comparison = (prompt | tool_llm | StrOutputParser()).invoke({
        "species1": species1, "context1": context1,
        "species2": species2, "context2": context2,
    })
    return {"content": comparison, "sources": extract_sources(docs1 + docs2)}





# ---- agent-as-tool: agent_1 can consult agent_2 -----------------------------------------
@tool
def consult_safety_analyst(species: str, region: str | None = None) -> dict:
    """Ask the safety_analyst agent to check conservation and invasive status for a species.
    Call this before finalizing any answer that touches on legal protection, invasive status,
    or conservation status, so that information comes from a dedicated check rather than a guess."""
    query = f"Check the conservation and invasive status for {species}"
    if region:
        query += f" in {region}"
    result = agent_2.invoke({"messages": [{"role": "user", "content": query}]})
    messages = result["messages"]
    
    sources: list[str] = []
    for msg in messages:
        if getattr(msg, "type", None) != "tool":
            continue
        content = getattr(msg, "content", None)
        try:
            payload = content if isinstance(content, dict) else json.loads(content)
        except (TypeError, ValueError):
            continue
        if isinstance(payload, dict):
            sources.extend(payload.get("sources", []))

    final_content = getattr(messages[-1], "content", None)
    if isinstance(final_content, list):  # responses-API content blocks
        final_content = "".join(
            block.get("text", "") for block in final_content if isinstance(block, dict)
        )
    return {"content": final_content or "No safety information could be retrieved.",
            "sources": sorted(set(sources))}




TOOLS_1 = [
    retrieve_documents,
    summarise_topic,
    compare_species,
    consult_safety_analyst,
]

print([t.name for t in TOOLS_1])


['retrieve_documents', 'summarise_topic', 'compare_species', 'consult_safety_analyst']


In [ ]:
# ---- BUILD AGENT 1 - wildlife_analyst ---- 
SYSTEM_PROMPT_1 = """You are a knowledgeable, safety-conscious naturalist assistant.

Your tools:
- retrieve_documents: fetches grounding passages from the field-guide database. Call this before
  answering any species-specific factual question -- answer only using what it returns, and say
  clearly if it doesn't support an answer.
- summarise_topic: Create a insighful text (with maximum of 2000 words) and final summary at the end of what the field guide says about a topic or species.
- compare_species: side-by-side comparison of two species using the field guide.
- consult_safety_analyst: a dedicated check for conservation/invasive status -- only call this,
  and only mention conservation/invasive/legal status in your answer, when the user's message
  actually asks about that or the instruction for this turn tells you to. Most questions
  (diet, habitat, appearance, behaviour, comparisons that don't mention status) should NOT
  touch this tool or this topic at all.

Answer ONLY the question the user actually asked -- do not pad every reply with the same
recurring sections. In particular:
- Do not add a "Conservation and Invasive Status" section, a numbered "Recommendations" list,
  or any conservation/invasive-status content unless the user's question or this turn's
  instruction is actually about that.
- Do not end the answer with "For more detailed information, you can refer to wikipedia" or any
  other stock sign-off. Only mention or link an external source if it is one of the sources
  retrieve_documents/summarise_topic/compare_species actually returned for this turn, and even
  then only when it adds something the user asked for.
- Vary your structure to match the question: a diet question gets a diet answer, a habitat
  question gets a habitat answer, not a repeat of the last message's template with one section
  swapped out.

Provide clear, and actionable answers. Do not claim an image-based identification is
verified.
"""

agent_1 = create_agent(
    model=llm,
    tools=TOOLS_1,
    system_prompt=SYSTEM_PROMPT_1,
    name="wildlife_analyst",
)

print(f"Agent 1 (wildlife_analyst) ready with {MODEL_NAME_improved} and {len(TOOLS_1)} tools ✅")


Agent 1 (wildlife_analyst) ready with gpt-5.6-luna and 4 tools ✅


In [ ]:
# ---- ORCHESTRATOR -- drives agent_1 ---- 
import json
from langchain_core.messages import AIMessageChunk, ToolMessage

# Tool names whose output carries citable sources, for the source list shown in the UI.
_SOURCE_BEARING_TOOLS = {"retrieve_documents", "summarise_topic", "compare_species",
                         "consult_safety_analyst"}

_MODEL_NODES = {"model", "agent"}


_INTERNAL_TAG = "internal"


DEBUG_STREAM = False


def _is_agent_1_reply(metadata: dict) -> bool:
    """True only for token chunks emitted by agent_1's own model node. Everything else that
    comes down the stream_mode='messages' channel -- inner tool LLMs, sub-agent graphs -- is
    intermediate computation and belongs in the thinking trace, not the answer."""
    if _INTERNAL_TAG in (metadata.get("tags") or []):
        return False
    return metadata.get("langgraph_node") in _MODEL_NODES


def generate_response(
    species: str | None,
    history: "HybridMemory",
    region: str | None = None,
    question: str | None = None,
    is_initial: bool | None = None,
) -> dict:
    """
    Build the next user turn, run agent_1 (it decides for itself whether/how to call its tools,
    including consulting agent_2), append the reply, and run the mandatory safety check.
    Returns the answer text and the sources any retrieval-style tool actually used.

    `history` is a HybridMemory instance (section 4), not a raw growing list: it exposes the
    system prompt, a rolling summary of everything older than the last `window_size` turns, and
    those recent turns verbatim. This keeps the tokens sent to agent_1 roughly flat as a chat
    goes on, instead of growing on every single turn the way a plain list would.

    Note: this path reads result["messages"], which only ever contains the *outer* graph's
    messages, so it is immune to the tool-internal leakage the streaming path has to guard
    against explicitly.
    """
    if is_initial is None:
        is_initial = not history.has_started()

    resolution = resolve_subject(question, active_subject=species, history=history,
                                 region=region) if question else {}
    active_species = resolution.get("subject") or species

    user_content = build_user_content(species, question, is_initial, resolution)
    history.add_message("user", user_content)

    messages = history.get_messages()
    result = agent_1.invoke({"messages": messages})
    new_messages = result["messages"][len(messages):]

    assistant_reply = None
    sources: list[str] = []
    for msg in new_messages:
        msg_type = getattr(msg, "type", None)
        msg_name = getattr(msg, "name", None)
        msg_content = getattr(msg, "content", None)

        if msg_type == "tool" and msg_name in _SOURCE_BEARING_TOOLS and msg_content:
            try:
                payload = msg_content if isinstance(msg_content, dict) else json.loads(msg_content)
                sources.extend(payload.get("sources", []))
            except (TypeError, ValueError):
                pass  # tool payload wasn't parseable JSON -- skip, don't fail the turn

        if msg_type == "ai" and msg_content:
            assistant_reply = _as_text(msg_content)

    if assistant_reply is None:
        assistant_reply = "Sorry, I wasn't able to put together an answer for that."
    sources = sorted(set(sources))

    # ---- Archive the assistant's raw reply ---- 
    history.add_message("assistant", assistant_reply)

    # ---- Mandatory safety warnings -- always run, never delegated to either agent's judgement ---- 
    warnings = safety_warnings(active_species or "", question or "", assistant_reply)
    if warnings:
        assistant_reply = f"{assistant_reply}\n\n" + "\n\n".join(warnings)

    return {
        'text': assistant_reply,
        'sources': sources,
        'species': active_species,
        'resolution': resolution,
    }


def generate_response_stream(
    species: str | None,
    history: "HybridMemory",
    region: str | None = None,
    question: str | None = None,
    is_initial: bool | None = None,
):
    """
    Streaming counterpart to generate_response, used by the /api/message/stream route
    (section 5) so the UI can render progressively instead of waiting for agent_1's whole
    turn to finish.

    Note on "thinking": gpt-4o-mini (the default MODEL_NAME) is not a reasoning model, so
    OpenAI doesn't return hidden chain-of-thought tokens for it -- there is no <think> block
    to surface. What this generator streams instead is agent_1's *real* intermediate steps as
    they happen: which tool it decided to call, and what that tool returned. That's genuine
    intermediate computation (not fabricated commentary), and it gives the UI the same
    "watch it think before it answers" experience.

    IMPORTANT -- stream_mode="messages" yields token chunks from *every* LLM call in the
    graph, including ones made inside tools. summarise_topic and compare_species each run
    their own LLM, and consult_safety_analyst runs agent_2's whole graph; all of those emit
    AIMessageChunks with no tool_calls, so a naive `isinstance(chunk, AIMessageChunk) and not
    chunk.tool_calls` test lets them through as answer text. When two such tools run in
    parallel their streams interleave token-by-token and the visible answer comes out
    shredded. _is_agent_1_reply() is what keeps them out.

    Yields dicts:
      {"type": "thinking", "text": "..."}                    -- one line of the tool-use trace
      {"type": "answer_delta", "text": "..."}                 -- a chunk of the final answer
      {"type": "done", "text": ..., "sources": [...], "species": ...}  -- final, exactly once
    """
    if is_initial is None:
        is_initial = not history.has_started()

    resolution = resolve_subject(question, active_subject=species, history=history,
                                 region=region) if question else {}
    active_species = resolution.get("subject") or species
    if resolution.get("subject_changed") and active_species:
        yield {"type": "thinking", "text": f"Subject is now {active_species}"}

    user_content = build_user_content(species, question, is_initial, resolution)
    history.add_message("user", user_content)
    messages = history.get_messages()

    answer_parts: list[str] = []
    sources: list[str] = []
    seen_tool_call_ids: set[str] = set()
    seen_tool_result_ids: set[str] = set()
    debug_seen = 0

    for chunk, metadata in agent_1.stream({"messages": messages}, stream_mode="messages"):
        metadata = metadata or {}
        node = metadata.get("langgraph_node")

        if DEBUG_STREAM and debug_seen < 12:
            debug_seen += 1
            print(f"[stream] node={node!r} ns={metadata.get('checkpoint_ns')!r} "
                  f"tags={metadata.get('tags')!r} type={type(chunk).__name__}")

        # --- thinking trace: a tool call has just been decided on -------------------------
        for call in (getattr(chunk, "tool_calls", None) or []):
            call_id, call_name = call.get("id"), call.get("name")
            if call_name and call_id and call_id not in seen_tool_call_ids:
                seen_tool_call_ids.add(call_id)
                prefix = "" if _is_agent_1_reply(metadata) else "safety_analyst: "
                yield {"type": "thinking", "text": f"{prefix}Calling {call_name}\u2026"}

        # --- thinking trace: a tool has finished running ----------------------------------
        if isinstance(chunk, ToolMessage):
            result_id = getattr(chunk, "tool_call_id", None) or id(chunk)
            if result_id not in seen_tool_result_ids:
                seen_tool_result_ids.add(result_id)
                tool_name = getattr(chunk, "name", None) or "tool"
                tool_content = _as_text(getattr(chunk, "content", None))
                preview = tool_content[:220] + ("\u2026" if len(tool_content) > 220 else "")
                yield {"type": "thinking", "text": f"{tool_name} returned: {preview}"}
                if tool_name in _SOURCE_BEARING_TOOLS and tool_content:
                    try:
                        payload = (json.loads(tool_content)
                                   if isinstance(tool_content, str) else tool_content)
                        sources.extend(payload.get("sources", []))
                    except (TypeError, ValueError):
                        pass  # tool payload wasn't parseable JSON -- skip, don't fail the turn
            continue  # a ToolMessage is never answer text

        # --- answer: agent_1's own tokens, and nothing else -------------------------------
        if not _is_agent_1_reply(metadata):
            continue
        if isinstance(chunk, AIMessageChunk) and not chunk.tool_calls:
            piece = _as_text(chunk.content)
            if piece:
                answer_parts.append(piece)
                yield {"type": "answer_delta", "text": piece}

    assistant_reply = "".join(answer_parts).strip()
    if not assistant_reply:
        assistant_reply = "Sorry, I wasn't able to put together an answer for that."
    sources = sorted(set(sources))

    # ---- Archive the assistant's raw reply ---- 
    history.add_message("assistant", assistant_reply)

    # ---- Mandatory safety warnings -- always run, never delegated to either agent's judgement ---- 
    warnings = safety_warnings(active_species or "", question or "", assistant_reply)
    final_text = assistant_reply
    if warnings:
        final_text = f"{assistant_reply}\n\n" + "\n\n".join(warnings)

    yield {"type": "done", "text": final_text, "sources": sources,
           "species": active_species}

## 4. Chat conversation memory

In [ ]:
import tiktoken


tokenizer = tiktoken.encoding_for_model("gpt-4o-mini")


def _as_text(content) -> str:
    """Normalize LangChain message content to plain text. content is usually a str, but
    with use_responses_api=True it can come back as a list of content blocks (e.g.
    [{"type": "text", "text": "..."}]) instead of a plain string."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict):
                parts.append(block.get("text", ""))
        return "".join(parts)
    return "" if content is None else str(content)



summary_prompt = ChatPromptTemplate.from_template(
    """Progressively update the running summary of this wildlife field-guide conversation.
Preserve concrete facts that may be asked about later -- species names, region, conservation /
invasive / legal-protection status, and anything the user decided or stated a preference about.
Keep it concise; this is background context for another agent, not the final answer.

Current summary:
{summary}

Message being archived ({role}):
{content}

Updated summary:"""
)
summary_update_chain = summary_prompt | llm | StrOutputParser()


class HybridMemory:
    """Hybrid conversation memory for `agent_1` (section 3).

    Keeps the last `window_size` messages verbatim (default: 3) and folds every older message
    into a single running summary via `summary_update_chain`, instead of the plain `history`
    list that used to be resent to `agent_1.invoke(...)` in full on every turn.

    Works directly with this app's message format -- {"role": ..., "content": ...} dicts, the
    same shape `generate_response` builds and `agent_1.invoke({"messages": ...})` expects --
    rather than the (human, ai) exchange tuples used for a plain single-turn chat elsewhere.
    """

    def __init__(self, system_prompt: str, window_size: int = 3):
        self.system_prompt = system_prompt
        self.summary = ""
        self.recent_messages: list[dict] = []   # verbatim {"role", "content"} dicts
        self.window_size = window_size
        self._has_user_message = False          # tracks whether the chat has started at all,
                                                  # even after early user turns get archived out

    def add_message(self, role: str, content: str):
        """Record one message and archive the oldest once the window overflows."""
        content = _as_text(content)
        if role == "user":
            self._has_user_message = True
        self.recent_messages.append({"role": role, "content": content})
        while len(self.recent_messages) > self.window_size:
            oldest = self.recent_messages.pop(0)
            self.summary = summary_update_chain.invoke({
                "summary": self.summary or "(none)",
                "role": oldest["role"],
                "content": oldest["content"],
            })

    def has_started(self) -> bool:
        """Replaces `any(msg['role'] == 'user' for msg in history)` -- that check breaks once
        old user turns have been archived out of `recent_messages`, so we track it separately."""
        return self._has_user_message

    def get_messages(self) -> list[dict]:
        """What `agent_1.invoke` actually sees: system prompt, an optional summary of the
        archived turns, then the verbatim recent window -- in that order."""
        messages = [{"role": "system", "content": self.system_prompt}]
        if self.summary:
            messages.append({
                "role": "system",
                "content": f"Summary of earlier conversation (for context only):\n{self.summary}",
            })
        messages.extend(self.recent_messages)
        return messages

    def token_count(self) -> int:
        """Rough token count of what actually gets sent -- for observability, mirrors the
        measurement the original notebook this was adapted from used."""
        text = self.system_prompt + self.summary + "".join(m["content"] for m in self.recent_messages)
        return len(tokenizer.encode(text))


## 5. Interactive RAG chat app


In [14]:
import json
import tempfile
import threading
import uuid
from flask import Flask, Response, jsonify, render_template_string, request, stream_with_context
from werkzeug.utils import secure_filename

app = Flask(__name__)
UPLOAD_DIR = Path(tempfile.gettempdir()) / 'inaturalist_rag_uploads'
UPLOAD_DIR.mkdir(exist_ok=True)

chats = {}

# --- Chat shell (static HTML) with adjusted textarea heights ---
PAGE = '''<!doctype html><html><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<title>The UK Field Guide</title>
<script src="https://cdn.jsdelivr.net/npm/marked/marked.min.js"></script>
<style>
:root{color-scheme:light}
*{box-sizing:border-box}
body{font:16px/1.45 system-ui,-apple-system,sans-serif;margin:0;background:#f4f7f2;color:#17351f;
     display:flex;flex-direction:column;min-height:100dvh}
header{padding:14px 16px;background:#28733d;color:#fff;text-align:center;flex:0 0 auto}
header h1{margin:0;font-size:1.15rem}
header p{margin:2px 0 0;font-size:.82rem;opacity:.9}
main{flex:1 1 auto;display:flex;flex-direction:column;max-width:720px;width:100%;margin:0 auto;
     padding:12px}
#upload-panel{background:#fff;padding:22px;border-radius:16px;box-shadow:0 6px 18px #0002;margin:auto}
#upload-panel label{display:block;margin-top:12px;font-weight:600;font-size:.92rem}
#upload-panel input, #upload-panel textarea{width:100%;padding:10px;margin-top:5px;border:1px solid #cfe0cf;border-radius:8px;font:inherit}
#upload-panel textarea{height:76px;resize:vertical}   /* ~3 lines by default; was 60px, then 40px min-height, then 28px single line */
#upload-panel button, .composer button{margin-top:14px;padding:11px 18px;border:0;border-radius:8px;
     background:#28733d;color:#fff;font-weight:700;cursor:pointer;font-size:1rem}
#upload-panel button:disabled, .composer button:disabled{opacity:.6;cursor:default}
.hint{font-size:.85rem;color:#446;margin-top:8px}
#chat-panel{display:none;flex-direction:column;flex:1 1 auto;min-height:0}
#species-bar{background:#fff;border:1px solid #dce8da;border-radius:12px;padding:10px 14px;
     margin-bottom:8px;font-size:.9rem;flex:0 0 auto}
#species-bar strong{color:#17351f}
.candidate-list{list-style:none;margin:6px 0 0;padding:0}
.candidate-item{display:flex;justify-content:space-between;gap:10px;padding:5px 0;
     border-top:1px dashed #dce8da;font-size:.88rem;color:#2a4a30}
.candidate-item:first-child{border-top:none}
.candidate-rank{font-weight:700;color:#17351f;margin-right:6px}
.candidate-name{flex:1}
.candidate-common{color:#5a7a5f;font-weight:400}
.candidate-confidence{font-weight:700;color:#28733d;white-space:nowrap}
#messages{flex:1 1 auto;padding:6px 2px;}
.msg{max-width:95%;margin:10px 0;padding:12px 14px;border-radius:14px;
     font-size:.96rem;line-height:1.4;box-sizing:border-box}
.msg.user{margin-left:auto;background:#28733d;color:#fff;border-bottom-right-radius:4px}
.msg.assistant{margin-right:0;width:100%;max-width:100%;background:#fff;border:1px solid #dce8da;
     border-bottom-left-radius:4px}
.msg p:first-child{margin-top:0}
.msg p:last-child{margin-bottom:0}
.sources{font-size:.8rem;color:#446;margin-top:8px;border-top:1px dashed #dce8da;padding-top:6px}
.sources-label{font-weight:600;margin-bottom:2px}
.sources-list{margin:0;padding-left:1.1em}
.sources-list li{margin:2px 0;word-break:break-word}
.source-link{color:#7ab8e0;text-decoration:underline;word-break:break-all}
.source-link:hover{color:#5aa0cc}
.source-link:visited{color:#9cc4dd}
.error{background:#ffe7e7;color:#7a1f1f;padding:10px 12px;border-radius:8px;font-size:.9rem;margin-top:8px}
.msg-body{line-height:1.5}
.msg-body p{margin:.5em 0}
.msg-body p:first-child{margin-top:0}
.msg-body p:last-child{margin-bottom:0}
.msg-body ul,.msg-body ol{margin:.4em 0;padding-left:1.3em}
.msg-body li{margin:.2em 0}
.msg-body li:last-child{margin-bottom:0}
.msg-body h1,.msg-body h2,.msg-body h3{margin:.7em 0 .3em}
.msg-body h1:first-child,.msg-body h2:first-child,.msg-body h3:first-child{margin-top:0}
.msg-body blockquote{margin:.4em 0;padding-left:.8em;border-left:3px solid #dce8da;color:#446}
.cursor{display:inline-block;color:#28733d;font-weight:700;animation:blink 1s steps(1) infinite}
@keyframes blink{50%{opacity:0}}
.thinking{margin:0 0 10px;font-size:.82rem}
.thinking summary{cursor:pointer;color:#28733d;font-weight:600;user-select:none}
.thinking summary.thinking-live{animation:blink 1s steps(1) infinite}
.thinking summary:hover{text-decoration:underline}
.thinking-list{margin:6px 0 0;padding-left:18px;color:#557;font-size:.8rem;line-height:1.55}
.thinking-list li{margin:2px 0;word-break:break-word}
.composer{flex:0 0 auto;display:flex;gap:8px;padding-top:8px;border-top:1px solid #dce8da;
     background:#f4f7f2;padding-bottom:env(safe-area-inset-bottom)}
.composer textarea{flex:1;resize:none;height:44px;padding:10px 12px;box-sizing:border-box;  /* was 44px/110px, then 36/80px min/max, then 28/60px min/max, then 26px fixed */
     border:1px solid #cfe0cf;border-radius:10px;font:inherit}
.composer button{margin-top:0;white-space:nowrap;height:44px;padding:0 18px;display:flex;align-items:center;justify-content:center;box-sizing:border-box}  /* matched to textarea height */
.restart{text-align:center;font-size:.85rem;margin-top:6px}
.restart a{color:#28733d}
.typing{font-size:.85rem;color:#688;padding:2px 4px}
</style></head>
<body>
<header><h1>The UK Field Guide</h1><p>Photograph a plant, insect, bird, or fungus, and chat about what it is and what to do.</p></header>
<main>
  <section id="upload-panel">
    <p><strong style="font-size: 1.5rem;">Ready to identify a wildlife specie?</strong> 
    <p>Upload a photo, ask a question, or both to get started! Feel free to add a region now, or skip it, I will check in later if I need one.</p>
    <label><strong>Photo</strong><span class="label-note"> (optional if you're just asking a question)</span><input id="photo" type="file" accept="image/*"></label>
    <label><strong>Region</strong><span class="label-note"> (optional)</span><input id="region" type="text" placeholder="e.g. England, UK"></label>
    <label><strong>Your question</strong><span class="label-note"> (optional if you're uploading a photo)</span><textarea id="question" rows="3" placeholder="e.g. Is this frog invasive in the UK?"></textarea></label>
    <button id="identify-btn">Start chat</button>
    <p class="hint">Identifications are model suggestions, not verified identifications. Please see the top-3 candidates once uploaded. No photo yet? I'll answer generally and you can add one anytime.</p>
    <div id="upload-error"></div>
  </section>

  <section id="chat-panel">
    <div id="species-bar"></div>
    <div id="messages"></div>
    <div id="chat-error"></div>
    <form class="composer" id="composer">
      <textarea id="followup" rows="1" placeholder="Ask a follow-up question…" required></textarea>
      <button id="send-btn" type="submit">Send</button>
    </form>
    <p class="restart"><a href="/">↺ Start over</a></p>
  </section>
</main>
<script>
let chatId = null;

function el(tag, cls, text){
  const e = document.createElement(tag);
  if(cls) e.className = cls;
  if(text !== undefined) e.textContent = text;
  return e;
}

function scrollMessagesToBottom(smooth) {
  // Page-level scroll: the whole document grows as messages/streamed text are added
  // (body is just min-height:100dvh, not bounded), so "scroll to bottom" means scrolling
  // the actual scrolling element of the document -- document.scrollingElement. That's the
  // standards-defined way to get the element the browser actually scrolls for the page
  // (it's <html> in standards mode; using this instead of window.scrollTo/document.body
  // avoids quirks-mode ambiguity). Reading .scrollHeight also forces the browser to lay
  // out any just-added/just-changed content first, so the target is always the CURRENT
  // bottom, not a stale one from before this render.
  //
  // We previously scrolled via restart.scrollIntoView() on the "Start over" link at the
  // bottom of the page. That added an extra layer of indirection (find a sibling element,
  // let the browser infer which ancestor to scroll and by how much) that didn't reliably
  // land at the true bottom. Scrolling the page's own scrollingElement directly to its own
  // scrollHeight is unambiguous -- there's no other element or container involved.
  //
  // Default to an instant jump ('auto'), not 'smooth': this runs on every streamed token
  // (via renderNow's rAF loop) and on every 'thinking' step (via addThinking) --
  // -- many times per second while an answer streams. A 'smooth' scroll is an animation;
  // firing a new one before the last one finishes restarts it, so a fast stream of calls
  // never lets any single animation complete and the page lags behind until streaming
  // stops. Pass smooth=true only for occasional, discrete jumps that have time to finish
  // uninterrupted (e.g. right after the user sends a message).
  const scroller = document.scrollingElement || document.documentElement;
  scroller.scrollTo({ top: scroller.scrollHeight, behavior: smooth ? 'smooth' : 'auto' });
}

// Auto-scroll is handled by explicit scrollMessagesToBottom() calls in the render
// functions below (renderMessage, addThinking, the streamed-answer renderer, finalize,
// showError) -- every path that adds visible content already scrolls itself. We
// deliberately do NOT also watch #messages with a MutationObserver: a blanket "scroll on
// any DOM change" watcher also fires on changes that aren't new content -- e.g. clicking
// "Show thinking" rewrites the summary's label text, which is a DOM mutation but not
// something that should yank the page down to the bottom.

// Renders sources as a labelled list rather than a single comma-joined string --
// individual citations can themselves contain commas (author lists, "Title, Publisher,
// Year"), so joining with ', ' makes it ambiguous where one citation ends and the next
// begins. One line per source is unambiguous regardless of what's inside each citation.
// format_citation() (see the ingestion/retrieval notebooks) always appends a URL, when
// it has one, as " — https://..." at the end of the citation string. This splits that off
// so the URL can be rendered as a real, clickable <a> instead of dead text -- everything
// before the separator stays as plain text exactly as returned.
const CITATION_URL_RE = / — (https?:\/\/\S+)$/;

// Builds one <li> for a citation: plain text for the "Title, Author, Year (notes)" part,
// a clickable <a class="source-link"> for the URL part if the citation has one.
function renderSourceItem(source){
  const item = document.createElement('li');
  const match = CITATION_URL_RE.exec(source);
  if(!match){
    item.textContent = source;
    return item;
  }
  const label = source.slice(0, match.index);
  const url = match[1];
  item.appendChild(document.createTextNode(label + ' — '));
  const link = document.createElement('a');
  link.href = url;
  link.textContent = url;
  link.className = 'source-link';
  link.target = '_blank';
  link.rel = 'noopener noreferrer';
  item.appendChild(link);
  return item;
}

function renderSources(container, sources){
  if(!sources || !sources.length) return;
  const wrap = el('div', 'sources');
  wrap.appendChild(el('div', 'sources-label', 'Sources:'));
  const list = el('ul', 'sources-list');
  for(const source of sources){
    list.appendChild(renderSourceItem(source));
  }
  wrap.appendChild(list);
  container.appendChild(wrap);
}

function renderMessage(msg){
  const wrap = el('div', 'msg ' + msg.role);
  const body = el('div', null);
  body.innerHTML = marked.parse(msg.text);
  wrap.appendChild(body);
  renderSources(wrap, msg.sources);
  document.getElementById('messages').appendChild(wrap);
  scrollMessagesToBottom();
}

// Creates an empty assistant bubble with a collapsible "Thinking…" panel (hidden until the
// first step arrives, closed by default -- the user expands it with a click) plus a body
// that's filled in as answer_delta events stream in. Returns handles the SSE reader below
// uses to update it live.
function renderStreamingAssistantMessage(){
  const wrap = el('div', 'msg assistant');

  const thinkingDetails = document.createElement('details');
  thinkingDetails.className = 'thinking';
  thinkingDetails.style.display = 'none';   // nothing to show until the first thinking step
  const thinkingSummary = document.createElement('summary');
  thinkingSummary.textContent = 'Thinking…';
  // Blinks (see the .thinking-live CSS rule) for as long as the turn is in progress --
  // removed in finalize()/showError() once the agent is done, at which point the label
  // switches to the static, non-blinking "Show/Hide thinking (N steps)" wording.
  thinkingSummary.classList.add('thinking-live');
  thinkingDetails.appendChild(thinkingSummary);
  const thinkingList = el('ul', 'thinking-list');
  thinkingDetails.appendChild(thinkingList);
  wrap.appendChild(thinkingDetails);

  const body = el('div', 'msg-body');
  const cursor = el('span', 'cursor', '▌');
  body.appendChild(cursor);
  wrap.appendChild(body);

  document.getElementById('messages').appendChild(wrap);
  scrollMessagesToBottom();

  let thinkingCount = 0;

  // SSE frames can arrive on localhost in fast bursts -- several answer_delta events can
  // land in a single network read and get processed in one synchronous JS pass, with no
  // browser paint in between. Re-parsing markdown and scrolling on every single delta
  // event would therefore just overwrite itself invisibly. Instead we only track the
  // latest full text on each delta, and do the (more expensive) markdown re-render +
  // scroll at most once per animation frame via requestAnimationFrame -- this guarantees
  // the browser actually paints between updates, so both the text and the scroll position
  // visibly progress as the answer streams in, roughly one rendered step per completed
  // word/frame rather than one silent jump at the very end.
  let latestText = '';
  let finalized = false;
  let frameScheduled = false;

  function renderNow(){
    frameScheduled = false;
    if(finalized) return;
    body.innerHTML = marked.parse(latestText || '');
    body.appendChild(cursor);
    scrollMessagesToBottom();   // <-- scroll to composer after each render
  }

  function scheduleRender(){
    if(frameScheduled) return;
    frameScheduled = true;
    requestAnimationFrame(renderNow);
  }

  return {
    addThinking(text){
      thinkingDetails.style.display = 'block';
      thinkingList.appendChild(el('li', null, text));
      thinkingCount += 1;
      // Label stays "Thinking…" (still blinking, via the thinking-live class set above) for
      // as long as the turn is running -- it only switches to the static step-count wording
      // once finalize() marks the turn done, so the blink genuinely tracks "still reasoning".
      scrollMessagesToBottom();   // <-- added here
    },
    updateAnswer(rawText){
      latestText = rawText;
      scheduleRender();
    },
    finalize(finalText, sources){
      finalized = true;
      cursor.remove();
      body.innerHTML = marked.parse(finalText);
      renderSources(wrap, sources);
      // The agent is done reasoning -- stop the blink and swap to the static, clickable
      // Show/Hide wording.
      thinkingSummary.classList.remove('thinking-live');
      if(thinkingCount > 0){
        const label = thinkingCount === 1 ? '1 step' : (thinkingCount + ' steps');
        thinkingSummary.textContent = 'Show thinking (' + label + ')';
      }
      scrollMessagesToBottom();   // <-- added here (no need for rAF / timeout)
    },
    showError(message){
      finalized = true;
      cursor.remove();
      // The turn ended (in failure) rather than genuinely finishing, but it's no longer
      // "still reasoning" either -- stop the blink the same way finalize() does.
      thinkingSummary.classList.remove('thinking-live');
      if(thinkingCount > 0){
        const label = thinkingCount === 1 ? '1 step' : (thinkingCount + ' steps');
        thinkingSummary.textContent = 'Show thinking (' + label + ')';
      }
      wrap.appendChild(el('div', 'error', message));
      scrollMessagesToBottom();
    }
  };
}

// Toggle the summary wording between Show/Hide as the user opens/closes the panel.
document.addEventListener('toggle', (evt) => {
  const details = evt.target;
  if(!details.classList || !details.classList.contains('thinking')) return;
  const summary = details.querySelector('summary');
  if(!summary) return;
  summary.textContent = summary.textContent.replace(/^(Show|Hide)/, details.open ? 'Hide' : 'Show');
}, true);

// Reads a POST'd Server-Sent-Events stream from /api/message/stream, parsing each
// "data: {...}\\n\\n" frame and handing the parsed event to onEvent as it arrives.
// Shared low-level SSE reader: parses "data: {...}\\n\\n" frames out of a fetch Response's
// body stream and hands each parsed event to onEvent as it arrives. Used by both
// streamIdentify (multipart POST, for the very first message) and streamAssistantReply
// (JSON POST, for every follow-up), so the initial identify+answer turn and every
// subsequent turn render through the exact same progressive "thinking" + streamed-text UI.
async function readEventStream(res, onEvent){
  if(!res.ok || !res.body){
    let message = 'Something went wrong.';
    try{ const data = await res.json(); message = data.error || message; }catch(e){}
    throw new Error(message);
  }
  const reader = res.body.getReader();
  const decoder = new TextDecoder();
  let buffer = '';
  while(true){
    const {value, done} = await reader.read();
    if(done) break;
    buffer += decoder.decode(value, {stream: true});
    let sepIndex;
    while((sepIndex = buffer.indexOf('\\n\\n')) !== -1){
      const rawEvent = buffer.slice(0, sepIndex);
      buffer = buffer.slice(sepIndex + 2);
      const dataLine = rawEvent.split('\\n').find(line => line.startsWith('data:'));
      if(!dataLine) continue;
      const jsonStr = dataLine.slice(5).trim();
      if(!jsonStr) continue;
      let parsedEvent;
      try{ parsedEvent = JSON.parse(jsonStr); }catch(e){ continue; }
      onEvent(parsedEvent);
    }
  }
}

async function streamAssistantReply(payload, onEvent){
  const res = await fetch('/api/message/stream', {
    method: 'POST',
    headers: {'Content-Type': 'application/json'},
    body: JSON.stringify(payload)
  });
  await readEventStream(res, onEvent);
}

// Same idea as streamAssistantReply but for the very first turn: posts the upload form
// (photo/region/question) to /api/identify/stream and streams back a "species" event
// (classification result) followed by the same thinking/answer_delta/done events as any
// other turn.
async function streamIdentify(formData, onEvent){
  const res = await fetch('/api/identify/stream', {
    method: 'POST',
    body: formData
  });
  await readEventStream(res, onEvent);
}

function formatConfidence(score){
  if(score === null || score === undefined || score === '' || isNaN(score)) return 'n/a';
  const value = Number(score);
  const pct = value <= 1 ? value * 100 : value;
  return pct.toFixed(1) + '%';
}

function renderSuggestions(species, suggestions, pending){
  const bar = document.getElementById('species-bar');
  bar.innerHTML = '';
  if(pending){
    bar.appendChild(el('strong', null, 'Identifying your photo…'));
    bar.appendChild(el('div', 'hint', 'This usually takes just a few seconds.'));
    return;
  }
  if(!species){
    bar.appendChild(el('strong', null, 'No photo has been uploaded.'));
    bar.appendChild(el('div', 'hint', "I'll answer based on the general RAG-grounded field guide knowledge."));
    return;
  }
  bar.appendChild(el('strong', null, 'Conclusively identified as ' + species + '!'));
  const label = el('div', 'hint', 'Classification result (scientific name, common name, confidence %)');
  bar.appendChild(label);
  const list = el('ol', 'candidate-list');
  suggestions.forEach((s, i) => {
    const item = el('li', 'candidate-item');
    const name = el('span', 'candidate-name');
    name.appendChild(el('span', 'candidate-rank', (i + 1) + '.'));
    name.appendChild(document.createTextNode(s.scientific_name));
    if(s.common_name){
      name.appendChild(el('span', 'candidate-common', ' (' + s.common_name + ')'));
    }
    const confidence = el('span', 'candidate-confidence', formatConfidence(s.combined_score));
    item.appendChild(name);
    item.appendChild(confidence);
    list.appendChild(item);
  });
  bar.appendChild(list);
}

document.getElementById('question').addEventListener('keydown', (evt) => {
  if(evt.key === 'Enter' && !evt.shiftKey){
    evt.preventDefault();
    const btn = document.getElementById('identify-btn');
    if(!btn.disabled) btn.click();
  }
});

document.getElementById('identify-btn').addEventListener('click', async () => {
  const photoInput = document.getElementById('photo');
  const questionInput = document.getElementById('question');
  const errorBox = document.getElementById('upload-error');
  errorBox.innerHTML = '';
  const hasPhoto = photoInput.files.length > 0;
  const question = questionInput.value.trim();
  if(!hasPhoto && !question){
    errorBox.appendChild(el('div', 'error', 'Add a photo, a question, or both to get started.'));
    return;
  }
  const fd = new FormData();
  if(hasPhoto) fd.append('photo', photoInput.files[0]);
  fd.append('region', document.getElementById('region').value);
  fd.append('question', question);
  document.getElementById('identify-btn').disabled = true;

  // Switch to the chat page immediately, before the network request even starts -- don't
  // make the user stare at a disabled button while classification + the agent's answer run.
  // The species bar opens in a "working on it" state and the assistant bubble opens with
  // its own live "thinking" panel right away, exactly like every follow-up turn does.
  chatId = null;
  document.getElementById('upload-panel').style.display = 'none';
  document.getElementById('chat-panel').style.display = 'flex';
  document.getElementById('messages').innerHTML = '';
  document.getElementById('chat-error').innerHTML = '';
  renderSuggestions(null, [], hasPhoto);
  if(question) renderMessage({role: 'user', text: question});

  const assistant = renderStreamingAssistantMessage();
  let answerText = '';

  try{
    await streamIdentify(fd, (evt) => {
      if(evt.type === 'species'){
        chatId = evt.chat_id;
        renderSuggestions(evt.species, evt.suggestions, false);
      }else if(evt.type === 'thinking'){
        assistant.addThinking(evt.text);
      }else if(evt.type === 'answer_delta'){
        answerText += evt.text;
        assistant.updateAnswer(answerText);
      }else if(evt.type === 'done'){
        assistant.finalize(evt.text, evt.sources);
      }else if(evt.type === 'error'){
        throw new Error(evt.text || 'Something went wrong.');
      }
    });
  }catch(err){
    assistant.showError(err.message);
    scrollMessagesToBottom();
    // Classification itself never even finished (no chatId yet) -- send the user back to
    // the upload screen with the error rather than stranding them on an empty chat.
    if(!chatId){
      document.getElementById('chat-panel').style.display = 'none';
      document.getElementById('upload-panel').style.display = 'block';
      errorBox.appendChild(el('div', 'error', err.message));
    }
  }finally{
    document.getElementById('identify-btn').disabled = false;
  }
});

document.getElementById('followup').addEventListener('keydown', (evt) => {
  if(evt.key === 'Enter' && !evt.shiftKey){
    evt.preventDefault();
    const btn = document.getElementById('send-btn');
    if(!btn.disabled) btn.click();
  }
});

document.getElementById('composer').addEventListener('submit', async (evt) => {
  evt.preventDefault();
  const textarea = document.getElementById('followup');
  const text = textarea.value.trim();
  if(!text || !chatId) return;
  const errorBox = document.getElementById('chat-error');
  errorBox.innerHTML = '';
  renderMessage({role:'user', text: text});
  textarea.value = '';
  document.getElementById('send-btn').disabled = true;

  const assistant = renderStreamingAssistantMessage();
  let answerText = '';

  try{
    await streamAssistantReply({chat_id: chatId, text: text}, (evt) => {
      // Scrolling/re-rendering is handled inside renderStreamingAssistantMessage's own
      // methods (throttled to one animation frame per update for answer_delta -- see
      // renderStreamingAssistantMessage above), so this callback just routes each event.
      if(evt.type === 'thinking'){
        assistant.addThinking(evt.text);
      }else if(evt.type === 'answer_delta'){
        answerText += evt.text;
        assistant.updateAnswer(answerText);
      }else if(evt.type === 'done'){
        assistant.finalize(evt.text, evt.sources);
      }else if(evt.type === 'error'){
        throw new Error(evt.text || 'Something went wrong.');
      }
    });
  }catch(err){
    assistant.showError(err.message);
    scrollMessagesToBottom();
  }finally{
    document.getElementById('send-btn').disabled = false;
  }
});
</script>
</body></html>'''

@app.route('/')
def home():
    return render_template_string(PAGE)


@app.route('/api/identify/stream', methods=['POST'])
def api_identify_stream():
    photo = request.files.get('photo')
    region = (request.form.get('region') or '').strip() or None
    question = (request.form.get('question') or '').strip() or None
    has_photo = bool(photo and photo.filename)

    if not has_photo and not question:
        return jsonify(error='Add a photo, a question, or both to get started.'), 400

    image_path = None
    if has_photo:
        image_path = UPLOAD_DIR / f"{uuid.uuid4().hex}_{secure_filename(photo.filename)}"
        photo.save(image_path)

    def event_stream():
        try:
            species, common_name, suggestions = None, None, []

            if image_path is not None:
                yield f"data: {json.dumps({'type': 'thinking', 'text': 'Classifying the photo…'})}\n\n"
                try:
                    suggestions, _ = identify_image(image_path, jwt_token, lat=None, lng=None, observed_on=None)
                except Exception as exc:
                    yield f"data: {json.dumps({'type': 'error', 'text': str(exc)})}\n\n"
                    return
                if not suggestions:
                    yield f"data: {json.dumps({'type': 'error', 'text': 'The classifier did not return a species suggestion.'})}\n\n"
                    return
                species = suggestions[0]['scientific_name']
                common_name = suggestions[0].get('common_name')

            # Register the chat as soon as we know the species (or lack of one) -- before
            # agent_1 even starts -- so a follow-up composer message still has somewhere to
            # go even if the streamed answer below fails partway through.
            chat_id = uuid.uuid4().hex
            history = HybridMemory(SYSTEM_PROMPT_1, window_size=3)  # section 4 hybrid memory
            display_messages = [{'role': 'user', 'text': question}] if question else []
            chats[chat_id] = {
                'species': species,
                'suggestions': suggestions,
                'region': region,
                'history': history,
                'display_messages': display_messages,
            }

            yield f"data: {json.dumps({'type': 'species', 'chat_id': chat_id, 'species': species, 'suggestions': suggestions})}\n\n"

            is_initial = bool(species) and not question

            for event in generate_response_stream(
                species=species,
                history=history,
                region=region,
                question=question,
                is_initial=is_initial,
            ):
                if event['type'] == 'done':
                    # Remember what the turn ended up being about, so the next "it" in this
                    # chat resolves against the right species.
                    chats[chat_id]['species'] = event.get('species') or chats[chat_id]['species']
                    display_messages.append({
                        'role': 'assistant',
                        'text': event['text'],
                        'sources': event['sources'],
                    })
                yield f"data: {json.dumps(event)}\n\n"
        except Exception as exc:
            yield f"data: {json.dumps({'type': 'error', 'text': str(exc)})}\n\n"
        finally:
            if image_path is not None:
                image_path.unlink(missing_ok=True)

    return Response(
        stream_with_context(event_stream()),
        mimetype='text/event-stream',
        headers={'Cache-Control': 'no-cache', 'X-Accel-Buffering': 'no'},
    )

# ------------------------------------------------------------------------------
# /api/message – unchanged
# ------------------------------------------------------------------------------
@app.route('/api/message', methods=['POST'])
def api_message():
    payload = request.get_json(silent=True) or {}
    chat_id, text = payload.get('chat_id'), (payload.get('text') or '').strip()
    chat = chats.get(chat_id)
    if not chat or not text:
        return jsonify(error='Your chat session or question is missing.'), 400
    try:
        species = chat['species']
        common_name = next((s.get('common_name') for s in chat['suggestions'] if s['scientific_name'] == species), None)

        result = generate_response(
            species=species,
            question=text,
            history=chat['history'],
            region=chat['region'],
            is_initial=False,
        )
        chat['species'] = result.get('species') or species
        display_message = {
            'role': 'assistant',
            'text': result['text'],
            'sources': result['sources'],
        }
        chat['display_messages'].append(display_message)
        return jsonify(messages=[display_message])
    except Exception as exc:
        return jsonify(error=str(exc)), 502

# ------------------------------------------------------------------------------
# /api/message/stream -- streams agent_1's tool-use "thinking" trace and its answer,
# piece by piece, over Server-Sent Events, instead of waiting for the whole turn to
# finish. Used by the chat composer (section 5 JS) so the reply appears progressively
# and the message pane can auto-scroll as it grows.
# ------------------------------------------------------------------------------
@app.route('/api/message/stream', methods=['POST'])
def api_message_stream():
    payload = request.get_json(silent=True) or {}
    chat_id, text = payload.get('chat_id'), (payload.get('text') or '').strip()
    chat = chats.get(chat_id)
    if not chat or not text:
        return jsonify(error='Your chat session or question is missing.'), 400

    species = chat['species']
    common_name = next((s.get('common_name') for s in chat['suggestions'] if s['scientific_name'] == species), None)

    def event_stream():
        try:
            for event in generate_response_stream(
                species=species,
                question=text,
                history=chat['history'],
                region=chat['region'],
                is_initial=False,
            ):
                if event['type'] == 'done':
                    chat['species'] = event.get('species') or chat['species']
                    display_message = {
                        'role': 'assistant',
                        'text': event['text'],
                        'sources': event['sources'],
                    }
                    chat['display_messages'].append(display_message)
                yield f"data: {json.dumps(event)}\n\n"
        except Exception as exc:
            yield f"data: {json.dumps({'type': 'error', 'text': str(exc)})}\n\n"

    return Response(
        stream_with_context(event_stream()),
        mimetype='text/event-stream',
        headers={'Cache-Control': 'no-cache', 'X-Accel-Buffering': 'no'},
    )

# ------------------------------------------------------------------------------
# Server lifecycle – unchanged
# ------------------------------------------------------------------------------
import os
import signal
import socket
import subprocess
import sys
import time
from werkzeug.serving import make_server

FIELD_GUIDE_PORT = 5050
FIELD_GUIDE_HOST = '127.0.0.1'

def _stop_previous_field_guide_server():
    state = globals().get('_field_guide_server_state')
    if not state:
        return
    try:
        state['server'].shutdown()
        state['server'].server_close()
    except Exception:
        pass
    state['thread'].join(timeout=5)
    globals()['_field_guide_server_state'] = None

def _port_is_free(host, port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
        probe.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        try:
            probe.bind((host, port))
            return True
        except OSError:
            return False

def _reclaim_port(port):
    reclaimed = False
    if sys.platform != 'win32':
        try:
            pids = subprocess.check_output(
                ['lsof', '-ti', f'tcp:{port}'], stderr=subprocess.DEVNULL
            ).decode().split()
            for pid in pids:
                os.kill(int(pid), signal.SIGTERM)
                reclaimed = True
        except Exception:
            pass
    if not reclaimed:
        try:
            import psutil
            for conn in psutil.net_connections(kind='inet'):
                if conn.laddr and conn.laddr.port == port and conn.pid:
                    try:
                        psutil.Process(conn.pid).terminate()
                        reclaimed = True
                    except Exception:
                        pass
        except Exception:
            pass
    return reclaimed

_stop_previous_field_guide_server()

if not _port_is_free(FIELD_GUIDE_HOST, FIELD_GUIDE_PORT):
    _reclaim_port(FIELD_GUIDE_PORT)
    time.sleep(0.5)

if not _port_is_free(FIELD_GUIDE_HOST, FIELD_GUIDE_PORT):
    raise RuntimeError(
        f'Port {FIELD_GUIDE_PORT} is still in use after trying to reclaim it. '
        'Something outside this kernel is holding it -- close that program '
        '(on macOS, check System Settings -> General -> AirDrop & Handoff -> '
        'AirPlay Receiver if FIELD_GUIDE_PORT is 5000), or run '
        '`pip install psutil` for more reliable auto-reclaim, then re-run this cell. '
        'You can also change FIELD_GUIDE_PORT above to a port you know is free.'
    )

_server = make_server(FIELD_GUIDE_HOST, FIELD_GUIDE_PORT, app, threaded=True)
_thread = threading.Thread(target=_server.serve_forever, name='FieldGuide-Flask', daemon=True)
_thread.start()
globals()['_field_guide_server_state'] = {'server': _server, 'thread': _thread}
print(f"App (re)started at http://{FIELD_GUIDE_HOST}:{FIELD_GUIDE_PORT} -- now serving this cell's code.")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


App (re)started at http://127.0.0.1:5050 -- now serving this cell's code.
